## Este notebook se utilizará para comprobar que noy hay rarezas en la base de datos.

#### Instalando dependencias

In [1]:
!pip install oracledb pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install oracledb pandas -q

import pandas as pd
import oracledb


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Parámetros
DB_USER = "PMAT_BTCH[PMATOWNER]"
DB_PASS = "w6IT%_)M>&"
DB_DSN  = "racdb-pre.si.unav.es:1521/UNSIDPRE.UNAV"  # ej.: db.mihost.com:1521/ORCLPDB1


In [5]:
# Carga de datos
conn = oracledb.connect(user=DB_USER, password=DB_PASS, dsn=DB_DSN)
sql = """
SELECT OPP_ID_ETAPA_COMP, OPP_ID, ETAPA, SUBETAPA, TARGET_REAL
FROM PMATOWNER.PMAT_PREDICTION
"""
df = pd.read_sql(sql, conn)
conn.close()
print("Dimensiones:", df.shape)
df.head(15)


C:\Users\malmendrosv\AppData\Local\Temp\2\ipykernel_14936\1217006939.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


Dimensiones: (51169, 5)


,OPP_ID_ETAPA_COMP,OPP_ID,ETAPA,SUBETAPA,TARGET_REAL
0,0066900001dHQILAA4__Inicio__NA,0066900001dHQILAA4,Inicio,NA,0
1,0066900001g51JkAAI__Información__NA,0066900001g51JkAAI,Información,NA,0
2,0066900001g51JkAAI__Inicio__NA,0066900001g51JkAAI,Inicio,NA,0
3,0066900001hU2nnAAC__Información__NA,0066900001hU2nnAAC,Información,NA,0
4,0066900001hU2nnAAC__Inicio__NA,0066900001hU2nnAAC,Inicio,NA,0
5,0066900001j7KF0AAM__Información__Recibida,0066900001j7KF0AAM,Información,Recibida,0
6,0066900001j7KF0AAM__Inicio__NA,0066900001j7KF0AAM,Inicio,NA,0
7,0066900001j7KF0AAM__Validación__Recibida,0066900001j7KF0AAM,Validación,Recibida,0
8,0066900001j7KF0AAM__Validación__Completa,0066900001j7KF0AAM,Validación,Completa,0
9,0066900001j7KF0AAM__Pruebas de admisión__Convo...,0066900001j7KF0AAM,Pruebas de admisión,Convocado,0


In [6]:
# Duplicados por OPP_ID_ETAPA_COMP
dup_comp = df[df.duplicated(subset=["OPP_ID_ETAPA_COMP"], keep=False)].sort_values("OPP_ID_ETAPA_COMP")
print(f"Filas duplicadas por OPP_ID_ETAPA_COMP: {len(dup_comp)}")
print(f"Claves OPP_ID_ETAPA_COMP afectadas: {dup_comp['OPP_ID_ETAPA_COMP'].nunique()}")
dup_comp.head(50)

# Duplicados por OPP_ID
dup_opp = df[df.duplicated(subset=["OPP_ID"], keep=False)].sort_values("OPP_ID")
print(f"Filas duplicadas por OPP_ID: {len(dup_opp)}")
print(f"IDs OPP_ID afectados: {dup_opp['OPP_ID'].nunique()}")
dup_opp.head(50)

# Resumen
print("Duplicados (OPP_ID_ETAPA_COMP):", df.duplicated(subset=["OPP_ID_ETAPA_COMP"]).sum())
print("Duplicados (OPP_ID):", df.duplicated(subset=["OPP_ID"]).sum())


Filas duplicadas por OPP_ID_ETAPA_COMP: 0
Claves OPP_ID_ETAPA_COMP afectadas: 0
Filas duplicadas por OPP_ID: 49440
IDs OPP_ID afectados: 7874
Duplicados (OPP_ID_ETAPA_COMP): 0
Duplicados (OPP_ID): 41566


In [9]:
# Columnas a evaluar
cols = ["OPP_ID_ETAPA_COMP", "OPP_ID", "ETAPA", "SUBETAPA"]

# Copia para análisis de vacíos (sin alterar NULLs reales)
df_norm = df.copy()
for c in cols:
    if c in df_norm.columns and df_norm[c].dtype == "object":
        df_norm[c] = df_norm[c].astype(str).str.strip()

# 1) NULLs (NaN/None) por columna
null_counts = df[cols].isna().sum().sort_values(ascending=False)
print("=== NULLs por columna ===")
print(null_counts.to_string())
print()

# 2) VACÍOS por columna (''/NA/N/A/None/null tras strip, pero SIN tocar NULLs)
vacuum_markers = {"", "NA", "N/A", "None", "null", "Null", "NULL"}
def is_empty_series(s):
    if s.dtype != "object":
        return pd.Series(False, index=s.index)
    stripped = s.astype(str).str.strip()
    # Solo contamos vacíos donde el valor original NO es null
    return (~s.isna()) & stripped.isin(vacuum_markers)

empty_counts = {c: is_empty_series(df_norm[c]).sum() for c in cols}
print("=== Vacíos por columna (''/NA/N/A/None/null) ===")
print(pd.Series(empty_counts).sort_values(ascending=False).to_string())
print()

# 3) Filas problemáticas en la clave
rows_key_null = df[df["OPP_ID_ETAPA_COMP"].isna()]
rows_key_empty = df_norm[is_empty_series(df_norm["OPP_ID_ETAPA_COMP"])]

print(f"Filas con OPP_ID_ETAPA_COMP NULL: {len(rows_key_null)}")
print(f"Filas con OPP_ID_ETAPA_COMP VACÍO: {len(rows_key_empty)}")
display(rows_key_null.head(20))
display(rows_key_empty.head(20))

# 4) Unicidad de la clave ignorando NULLs vs ignorando VACÍOS
dups_ign_nulls = df[~df["OPP_ID_ETAPA_COMP"].isna()].duplicated(subset=["OPP_ID_ETAPA_COMP"]).sum()
mask_valid_key = ~is_empty_series(df_norm["OPP_ID_ETAPA_COMP"])
dups_ign_empties = df_norm[mask_valid_key].duplicated(subset=["OPP_ID_ETAPA_COMP"]).sum()

print("=== Unicidad de OPP_ID_ETAPA_COMP ===")
print(f"Duplicados ignorando NULLs: {dups_ign_nulls}")
print(f"Duplicados ignorando VACÍOS: {dups_ign_empties}")


=== NULLs por columna ===
OPP_ID_ETAPA_COMP    0
OPP_ID               0
ETAPA                0
SUBETAPA             0

=== Vacíos por columna (''/NA/N/A/None/null) ===
OPP_ID_ETAPA_COMP    0
OPP_ID               0
ETAPA                0
SUBETAPA             0

Filas con OPP_ID_ETAPA_COMP NULL: 0
Filas con OPP_ID_ETAPA_COMP VACÍO: 0


,OPP_ID_ETAPA_COMP,OPP_ID,ETAPA,SUBETAPA,TARGET_REAL


,OPP_ID_ETAPA_COMP,OPP_ID,ETAPA,SUBETAPA,TARGET_REAL


=== Unicidad de OPP_ID_ETAPA_COMP ===
Duplicados ignorando NULLs: 0
Duplicados ignorando VACÍOS: 0


In [10]:
# Verificar longitudes y espacios raros
cols = ["OPP_ID_ETAPA_COMP","OPP_ID","ETAPA","SUBETAPA"]
for c in cols:
    if c in df.columns and df[c].dtype=="object":
        print(f"{c} max_len:", df[c].astype(str).str.len().max())
        print(f"{c} leading/trailing spaces:", (df[c].astype(str)!=df[c].astype(str).str.strip()).sum())


In [11]:
mask_bad = df["OPP_ID_ETAPA_COMP"].astype(str).str.contains(r"[^A-Za-z0-9_\-\.]", na=False)
print("Claves con caracteres raros:", mask_bad.sum())
df.loc[mask_bad, ["OPP_ID_ETAPA_COMP","OPP_ID","ETAPA","SUBETAPA"]].head(20)


Claves con caracteres raros: 41509


,OPP_ID_ETAPA_COMP,OPP_ID,ETAPA,SUBETAPA
1,0066900001g51JkAAI__Información__NA,0066900001g51JkAAI,Información,NA
3,0066900001hU2nnAAC__Información__NA,0066900001hU2nnAAC,Información,NA
5,0066900001j7KF0AAM__Información__Recibida,0066900001j7KF0AAM,Información,Recibida
7,0066900001j7KF0AAM__Validación__Recibida,0066900001j7KF0AAM,Validación,Recibida
8,0066900001j7KF0AAM__Validación__Completa,0066900001j7KF0AAM,Validación,Completa
9,0066900001j7KF0AAM__Pruebas de admisión__Convo...,0066900001j7KF0AAM,Pruebas de admisión,Convocado
10,0066900001j7KF0AAM__Pruebas de admisión__Prueb...,0066900001j7KF0AAM,Pruebas de admisión,Pruebas calificadas
11,0066900001j7KF0AAM__Estudio Centro__Recibida,0066900001j7KF0AAM,Estudio Centro,Recibida
12,0066900001j7KF0AAM__Estudio Centro__Resuelta,0066900001j7KF0AAM,Estudio Centro,Resuelta
13,0066900001j7KF0AAM__Propuesta centro__Recibida,0066900001j7KF0AAM,Propuesta centro,Recibida
